#### This code is  for plotting the heatmaps (VthCa Vs Inhibition) and study the effect of hysteresis with different inhibitory profiles, ...  (we want to replicate the effect of removing neuromodulation when the inhibition is increased)

In [ ]:
using Plots, Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, JLD2, Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, ForwardDiff, LinearAlgebra

folder_path = "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/functions_optim"
for file in readdir(folder_path)
    if endswith(file, ".jl")
        full_path = joinpath(folder_path, file)
        try
            include(full_path)
        catch err
            @warn "Error including file $(full_path)" exception=err
        end
    end
end


## Load parameters of the motoneurons

In [ ]:
######################  S-TYPE ##########################
#########################################################
nr_s = 10
# JLD2.@load "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/hystprofile_NEW.jld2" γn_new
# JLD2.@load "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/VthCa_new.jld2" VthCa_new

# Radius and soma lengths
rheobase_S = LinRange(3.5,6.5,nr_s) .+ rand(Normal(0.0,0.05),nr_s)  
rs_S = LinRange(38.75,41.25,nr_s)./10000 
ls_S = LinRange(77.5,82.5,nr_s)./10000 
rms_S = LinRange(1.15, 1.05,nr_s).*1000
rd_S = LinRange(20.75,31.25,nr_s)./10000 
ld_S = LinRange(5500.,6800.,nr_s)./10000 
rmd_S = LinRange(14.4,10.70,nr_s).*1000 

#  Maximun conductances for each of the ION channels mS/cm^2 
max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S =LinRange(16,25,nr_s) 
max_gca_S = LinRange(0.038,0.029,nr_s) 


# Rate constants of the kinetic processes in ms^-1
αm = ones(nr_s)*22.0
βm = ones(nr_s)*13.0

αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4

αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1

αq_S = ones(nr_s)*1.5
βq_S = LinRange(0.025,0.038,nr_s) 

αp_D = 0.008
βp_D = LinRange(0.014,0.016,nr_s) 


Apeak_S = LinRange(0.103,0.123,nr_s)
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);


#EXCITATORY PROFILE
amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000

# at some point delete these things
γa=1.;
γh = ones(nr_s)#γn_new
IF= LinRange(1.,1.,nr_s);  #factor for the inhibtion

#reversal potentials
EL=0; ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 

#maximun conductances
Ge=0.6; Gi=0.5 ; 

#initial voltages
Vd0 = 0.; 
Vs0 = 0.;

#refractory period
rp=5.0; 

const step = 0.05;
_int(num) = floor(Int, num);

#Pulse duration [mS]
pd=1; 

ri = 70.  # Ω.cm 3intracellular resistivity
cm = 1.; #membrane capacitance

Vthca_S= ones(nr_s);

MN_list=[];
Vth_S = zeros(nr_s); gls_S = zeros(nr_s); gld_S = zeros(nr_s); gc_S = zeros(nr_s); gna_S = zeros(nr_s); gkf_S = zeros(nr_s); gks_S = zeros(nr_s); gca_S = zeros(nr_s); rn_S = zeros(nr_s); cs_S = zeros(nr_s)  ; cd_S = zeros(nr_s)
for i in 1:nr_s 
    ##assuming that the somatic membrane can be modeled as a cylinder and using the defined parameters from the previous section
    gls_S[i] = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6  # conductance of the soma for the ith neuron (uS)
    gld_S[i] = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6  # # conductance of the dendrite for the ith neuron (uS)
    gc_S[i] = ((2/(((ri*ld_S[i])/(π*rd_S[i]^2))+((ri*ls_S[i])/(π*rs_S[i]^2)))))*1e6    # coupling conductance in uS
    rn_S[i] = 1/(gls_S[i]+((gld_S[i]*gc_S[i])/(gld_S[i]+gc_S[i]))) # input resistance of the neuron in MΩ     
    Vth_S[i] = round(rn_S[i]*rheobase_S[i],digits=2)   # in mV
    cs_S[i] = 2*π*ls_S[i]*rs_S[i]*cm*1e3   #inner capacitance of the soma nF
    cd_S[i] = 2*π*ld_S[i]*rd_S[i]*cm*1e3   #nF
    
    #conductances uS:
    gna_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3          
    gkf_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3     
    gks_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3     
    gca_S[i] = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3     

    S_MN = Dict(:Cs => cs_S[i], :Cd => cd_S[i], :Irh => rheobase_S[i], :gc => gc_S[i], :gNa => gna_S[i], :gKs => gks_S[i], :gKf => gkf_S[i], :gCa => gca_S[i], :gld => gld_S[i], :gls => gls_S[i], :αp => αp_D, :βp => βp_D[i], 
        :αq => αq_S[i], :βq => βq_S[i], :αm => αm[i], :βm => βm[i], :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i], :Vth => Vth_S[i], :VthCa => Vthca_S[i], :Rn => rn_S[i],
        :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh=>γh[i] , "type" => "All Type")
 
    MN_list = [MN_list;S_MN];
end
MNs = MN_list;


tspan = 0:0.05:sim_length; #time span of simulation defined by step size and length of simulation // change this to get more or less points in the simulation
m=0.0;h=1.0;n=0.0;q=0.0;p=0.0;m0=0.0;h0=1.0;n0=0.0;q0=0.0;p0=0.0; t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s) 
sim_time = _int(sim_length/0.05)   # because the step size is 0.05ms
spike_arr = zeros((nr_s, sim_time)) #storage
vs_arr = zeros((nr_s, sim_time)); #storage
pt_arr = zeros((nr_s, sim_time)); #storage
mt_arr = zeros((nr_s, sim_time)); #storage
ht_arr = zeros((nr_s, sim_time)); #storage
nt_arr = zeros((nr_s, sim_time)); #storage
qt_arr = zeros((nr_s, sim_time)); #storage
commoninput = tr; #needed for the run_sim() function


full_input = zeros((nr_s, sim_length))
independantnoise = zeros((nr_s, sim_length))
factor= 0.1 #for creating the inhibitory signal
inh_input=fill(factor*amplitude, sim_length);


In [ ]:
#########################################################################################
################## CHANGING VALUES OF NEUROMOD AND CONSTANT INHIBITION ##################
#########################################################################################

vector_vthca=  LinRange(0,10,10)   
vector_inh=  LinRange(0.,1.,10)
hysteresis_for_each_inh= zeros(10,10)
n=1

for j in 1:length(vector_vthca)
    for k in 1:length(vector_inh)
        
        vthca= vector_vthca[j] 
        factor= vector_inh[k]        
        inh_input=fill(factor*amplitude, sim_length)
     

        MN_list[n][:VthCa] = vthca

        MNs= MN_list

        
        spike_array, vs_array = run_sim_inh(n, inh_input);


        amplitude=0.4;
        sim_length=560000 ;
        half_triang = div(sim_length, 2);  
        flat_line = fill(-0.1, 20000);
        up = range(-0.1, amplitude, length = half_triang + 1)  ;
        down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
        tr = vcat(flat_line, up[2:end], down[2:end], flat_line);

        
        
        if any(spike_array[1, :] .== 1)
            
            value_1= tr[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
            value_2=  tr[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

            hysteresis_for_each_inh[j,k]= ((value_2/value_1).-1)[1]
        else
            hysteresis_for_each_inh[j,k]=100
        end

    end

end


In [ ]:
################################################################################
################## CHANGING VALUES OF NEUROMOD AND INHIBITION ##################
################################################################################

vector_vthca=  LinRange(0,10,10)   
vector_inh=  LinRange(0.,1.,10)
hysteresis_for_each_inh= zeros(10,10)
n=2


for j in 1:10
    for k in 1:10
        
        vthca= vector_vthca[j] 

        factor= vector_inh[k]        
        
        MN_list[n][:VthCa] = vthca

        MNs= MN_list

        # balanced
        # amplitude=0.4*factor
        # tr_length=28000
        # half_triang = div(tr_length, 2)  
        # up = range(0, amplitude, length = half_triang + 1)  
        # down = range(amplitude, 0, length = tr_length - half_triang + 1)  

        # pushpull
        amplitude=0.4
        a=0.4*factor
        tr_length=30000
        half_triang = div(tr_length, 2)  
        if k!=1
            down = range(amplitude, a, length = half_triang + 1)  
            up = range(a, amplitude, length = tr_length - half_triang + 1)  

        else
            down = range(0, 0, length = half_triang + 1)  
            up = range(0, 0, length = tr_length - half_triang + 1)  
        end
        inh_input = vcat(down[2:end], up[2:end])

        
        spike_array, vs_array = run_sim_inh(n, inh_input);


        amplitude=0.4;
        sim_length=560000 ;
        half_triang = div(sim_length, 2);  
        flat_line = fill(-0.1, 20000);
        up = range(-0.1, amplitude, length = half_triang + 1)  ;
        down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
        tr = vcat(flat_line, up[2:end], down[2:end], flat_line);

        if any(spike_array[1, :] .== 1)
            
            value_1= tr[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
            value_2=  tr[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

            hysteresis_for_each_inh[j,k]= ((value_2/value_1).-1)[1]
        else
            hysteresis_for_each_inh[j,k]=100
        end

    end

end


In [ ]:
using Plots

hysteresis_for_each_inh_modified = copy(hysteresis_for_each_inh)  
hysteresis_for_each_inh_modified[hysteresis_for_each_inh .> 99] .= NaN

# Flip the rows by reversing the matrix along the first dimension
hysteresis_for_each_inh_modified_flipped = reverse(hysteresis_for_each_inh_modified, dims=1)

# Plot the heatmap with flipped rows
Plots.heatmap(hysteresis_for_each_inh_modified_flipped,
    color=:inferno, 
    aspect_ratio=1, 
    grid=false, 
    xticks=false, 
    yticks=false, 
    xlabel="", 
    ylabel="", 
    frame=:box,
    clim=(-0.8, 0)  # Set the color limits from -0.75 to 0
)


In [ ]:
# JLD2.@load "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Heatmap_values/hysteresis_heatmap1_balanced_gamma1.jld2" hysteresis_for_each_inh

In [ ]:
JLD2.@save "/Users/nmt121/Desktop/Repositories/Motoneuron_Model/Data_MN_Model/Heatmap_values/hysteresis_heatmap1_pushpull_FINAL2.jld2" hysteresis_for_each_inh